# DS4DS Exercise Sheet 8

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.10.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

## Exercise 1 - Automatic and Approximate Derivatives

As some of you might have noticed with the last exercise sheet, 
manual differentiation of complicated functions tends to be error-prone and requires
knowledge of the inner workings of said functions.
Luckily, there are ways to avoid hand-crafted derivatives.
If you have Julia functions performing tedious computations, you can use one of the many
*Automatic Differentiation*  (AD) packages.
A good AD package gives exact derivatives for differentiable functions, and oftentimes they 
employ clever tricks to do so fast and in a memory-efficient way.
In case your function relies on external software (e.g., simulations), you could try to
approximate the gradients.
Traditionally, *Finite Differences* are the go-to approach for gradient approximations.

In this exercise, we are going to implement a finite difference scheme for multi-variate functions
ourselves.
Moreover, we take a look at the AD packages `FineteDiff`, `ForwardDiff` and `Zygote`.

### Manual Finite Differences


We are interested in the (first-order) partial derivatives of $\mathcal L\colon ℝ^q \to ℝ.$

The [finite difference operator](https://en.wikipedia.org/wiki/Finite_difference) $Δ_h^{w_i}$ 
with grid-size $h > 0$ maps $\mathcal L$ to $Δ_h^{w_i}[\mathcal L]$, and we would like 
$Δ_h^{w_i}[\mathcal L](\mathbf w) \approx ∂_{w_i} \mathcal L(\mathbf w)$.
That is, the operator should approximate the partial derivative of $\mathcal L$ with respect to $w_i$ 
at $\mathbf w \in ℝ^q$.
If $\mathcal L$ is two-times continuously differentiable, then the central finite difference scheme 
is of order 2, meaning that the error of the first-order derivatives is $\mathcal O(h^2)$.

There are, of course, alternatives: forward and backward schemes, and schemes of higher 
accuracy or order, see [this table in the Wikipedia](https://en.wikipedia.org/wiki/Finite_difference_coefficient).
For now, we want to stick with the central finite diffence scheme:
$$
Δ_h^{w_i}[\mathcal L](\mathbf w) = \frac{\mathcal L(\mathbf w + h \mathbf e_i) - \mathcal L(\mathbf w - h \mathbf e_i)}{2 h}.
$$
The $i^{\text{th}}$ unit vector $\mathbf e_i$ is all zeros, except at position $i$, where it has entry $1$.

### Exercise 1.a)
Write a function `finite_diff_grad` that takes a function `func` ($\mathcal L$), 
an input vector `w` and 
a scalar grid-size `h`, and returns the central finite difference gradient approximation 
$Δ_h[\mathcal L](\mathbf w) = [Δ_h^{w_1}[\mathcal L](\mathbf w), \ldots, Δ_h^{w_q}[\mathcal L](\mathbf w)]^T \in ℝ^q$. \
Do so by completing the cell below:

In [ ]:
"""
    finite_diff_grad(func, w::AbstractVector{W}, h::H=0.0001f0) where {W<:Real, H<:Real}

Return the finite difference gradient approximation of `func` at input `w`.
"""
function finite_diff_grad(func, w, h=0.0001)
    @assert h > 0 "Grid-size must be positive."

    # pre-allocate solution array:
    dw_func = zeros(length(w))
    
    # Note:
    # `zero(w)` gives an array with same element type and length
    # but above, `dw_func` is a `Vector{Float64}` to keep things simple
    
    # Now, fill `dw_func` to contain the finite difference approximations:    
    ### BEGIN SOLUTION

    ### END SOLUTION

    return dw_func
end

finite_diff_grad

We can test the function in the cell below:

In [ ]:
let # introduce a local scope to not accidentally pollute the global environment
    func(w) = sum(w .^ 2)

    w_one = ones(2)

    dw_func = finite_diff_grad(func, w_one)
    @assert length(dw_func) == length(w_one)
    @assert dw_func ≈ [2, 2]
    
    # compute gradient of simple uni-variate function:
    @assert finite_diff_grad( w -> w[1]^2, [1.0,] ) ≈ [2.0,]

    ### BEGIN HIDDEN TESTS
    # Did students follow the instructions and allocate right arrays without screwing the types?
    w_rand3 = rand(3)
    dw_func = finite_diff_grad(func, w_rand3)
    @assert length(dw_func) == length(w_rand3)
    @assert dw_func ≈ 2 .* w_rand3

    # How good are the gradients?
    func_lin(w) = sum(w)   # finite differences are exact for linear functions 
    w_rand5 = rand(5)
    dw_func = finite_diff_grad(func_lin, w_rand5)
    @assert length(dw_func) == length(w_rand5)
    @assert all(dw_func .≈ 1)

    dw_func = finite_diff_grad(func_lin, w_rand5, 1)
    @assert length(dw_func) == length(w_rand5)
    @assert all(dw_func .≈ 1)

    func_symmetric(w) = sum(cos.(w)) # central difference vanishes around symmetries
    dw_func = finite_diff_grad(func_symmetric, zeros(3))
    @assert sum(dw_func .^ 2) <= 1e-6
    ### END HIDDEN TESTS
end

### Exercise 1.b)
Investigate the accuracy of `finite_diff_grad` for different grid-sizes `h` on the function
$$ \mathcal L\colon ℝ^q \to ℝ, \; \mathcal L(\mathbf w) = \exp(w_1) + \sum_{i=1}^q w^4_i,$$
where $\mathbf w = [w_1, …, w_q]^T$.

Complete the code in the cell below to test the values $h\in \{10^{-1}, 10^{-2}, …, 10^{-10}\}$.
For each grid-size value $h$, store the error
$$
\left\| Δ_h[\mathcal L](\mathbf w) - \nabla \mathcal L(\mathbf w) \right\|^2
$$
in `fd_grad_errors`. That is, besides the finite-difference approximation you also have to calculate the analytical derivative!

In [ ]:
test_func(w) = exp(w[1]) + sum(w .^ 4)
function grad_test_func(w)
    global test_func
    ## return the **true** gradient vector of `test_func` w.r.t. `w`
    ### BEGIN SOLUTION
  
    ### END SOLUTION
end

# grid sizes ``h`` to test:
fd_grid_sizes = nothing
# Change `fd_grid_sizes` to a vector conforming to the exercise statement:
### BEGIN SOLUTION

### END SOLUTION

# evaluation point, **don't** change
w_fd = [π, 10, -ℯ / 20]

# exact gradient vector `grad_test_func_exact` at `w_fd`
grad_test_func_exact = nothing
# Change `grad_test_func_exact` to hold the true gradient:
### BEGIN SOLUTION

### END SOLUTION


# pre-allocate array to store the error values in
fd_grad_errors = zero(fd_grid_sizes)
for (i, h) in enumerate(fd_grid_sizes)
    ## compute finite difference gradient and store error in `fd_grad_errors[i]`
    ### BEGIN SOLUTION

    ### END SOLUTION 
end

We can have a look at the error values by plotting them, for example with `Makie`.
Usually, we would expect the error to decrease initially with the grid-size.
But at some point, round-off errors will lead to an increase again.
We can see this in this pre-made plot:
![fd error plot](fd.png)

### Exercise 1.c)

#### Intro

The Julia ecosystem provides many tools for [Automatic Differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation).
`Zygote` is used in the popular machine learning library [`Flux.jl`](https://fluxml.ai/Flux.jl/stable/) 
and implements reverse accumulation to compute loss function gradients.
`ForwardDiff` is a forward-mode library and very robust. It works on functions acting on 
`Real` input.
Lastly, `FiniteDiff` does as the name suggests and computes finite difference derivative
approximations.

<div class="alert alert-block alert-info">
Flux and Lux are destined to move to Enzyme at some point in time.
The project <a href="https://github.com/JuliaDiff/AbstractDifferentiation.jl">AbstractDifferentiation</a>
wants to provide a unified API for many differentiation packages, but lacks caching mechanisms
and is thus not endorsed by SciML libraries (yet).
</div>

In [9]:
import ForwardDiff as FD
import FiniteDiff
import Zygote

Use all the above libraries to compute partial first-order derivatives of the scaled
Gaussian RBF 
$$
m(\mathbf z; a, b, \mathbf c) = b \cdot \exp\left( -\frac{\| \mathbf z - \mathbf c \|^2}{a^2} \right).
$$
The function $m$ (like $m$odel) maps $\mathbf z\in ℝ^n$ to a scalar value.
It is **c**entered at $\mathbf c \in ℝ^n$, and has shape parameter $a>0$, and scaling factor $b \in ℝ$.

In [6]:
function rbf(z, a, b, c)
    r_squared = sum((z .- c) .^ 2)
    return b * exp(-r_squared / a^2)
end

rbf (generic function with 1 method)

#### Exercise

Use `FD.gradient`, `Zygote.gradient` and `FiniteDiff.finite_difference_derivative` to
calculate the partial derivatives of $m$ with respect to $\mathbf z$,
$\mathbf c$ and $a$ respectively:
* `FD.gradient`: $\partial m / \partial \mathbf z \in \mathbb{R}^2,$
* `Zygote.gradient`: $\partial m / \partial \mathbf c \in \mathbb{R}^2,$
* `FiniteDiff.finite_difference_derivative`: $\partial m / \partial a \in \mathbb{R}.$

To do so, you can use anonymous functions, e.g. `_z -> rbf(_z, c, a)`. 
**Always return a `Vector`!**

In [ ]:
function dz_rbf_ForwardDiff(z, a, b, c)
    ## compute partial derivative of `rbf` with respect to `z` using `ForwardDiff`,
    ## return a vector:
    ### BEGIN SOLUTION
    
    ### END SOLUTION
end

dz_rbf_ForwardDiff (generic function with 1 method)

In [ ]:
function dc_rbf_Zygote(z, a, b, c)
    ## compute partial derivative of `rbf` with respect to `c` using `Zygote`,
    ## return a vector...

    ## Take care: `Zygote.gradient` returns a tuple for each variable array you provide as argument!
    ### BEGIN SOLUTION
   
    ### END SOLUTION
end

dc_rbf_Zygote (generic function with 1 method)

In [ ]:
function da_rbf_FiniteDiff(z, a, b, c)
    ## FiniteDiff does not like the independent variable to be an Integer...
    ## So we cast it it to `Float64`
    a = Float64(a)
    
    ## compute partial derivative of `rbf` with respect to `a` using `FiniteDiff`,
    ## return a vector...
    ### BEGIN SOLUTION
    
    ### END SOLUTION
end

da_rbf_FiniteDiff (generic function with 1 method)

We should now be able to evaluate the partial gradients at 
$(\mathbf z, \mathbf c, a) = (\mathbf 1, \mathbf 0, 1)$:

In [ ]:
let z = ones(2), c = zeros(2), a = 1, b = 1
    ## evaluate partial gradient with respect to x
    @assert dz_rbf_ForwardDiff(z, a, b, c) isa Vector
    @assert length(dz_rbf_ForwardDiff(z, a, b, c)) == 2
    ### BEGIN HIDDEN TESTS
    all(isapprox(g, 0.1; rtol=1e-1) for g = dz_rbf_ForwardDiff([1, 2], 3, 4, [5, 6]))
    ### END HIDDEN TESTS
end

true

In [11]:
let z = ones(2), c = zeros(2), a = 1, b = 1
    @assert dc_rbf_Zygote(z, a, b, c) isa Vector
    @assert length(dc_rbf_Zygote(z, a, b, c)) == 2
    ### BEGIN HIDDEN TESTS
    all(isapprox(g, -0.1; rtol=1e-1) for g = dc_rbf_Zygote([1, 2], 3, 4, [5, 6]))
    ### END HIDDEN TESTS
end

true

In [12]:
let z = ones(2), c = zeros(2), a = 1, b = 1
    @assert da_rbf_FiniteDiff(z, a, b, c) isa Vector
    @assert length(da_rbf_FiniteDiff(z, a, b, c)) == 1
    ### BEGIN HIDDEN TESTS
    only(da_rbf_FiniteDiff(z, a, b, c)) ≈ 0.5413411329152712
    ### END HIDDEN TESTS
end

true

### Exercise 1d)

#### Intro

Just like on the last exercise sheet, we now compose a complete RBF approximation model 
$h \colon ℝ^n \to ℝ$
as the sum of $N_\text{RBF} \in ℕ$ RBFs with different parameters
$$
h(\mathbf z) = 
h(\mathbf z; \mathbf w)
= \sum_{i=1}^{N_{\text{RBF}}} m(\mathbf z; a_i, b_i, \mathbf c_i).
$$
Here, $\mathbf w$ is the complete (flattened) parameter vector of $h$ holding 
$a_i, b_i$ and $\mathbf c_i$ for $i=1,…, N_{\text{RBF}}$.

In [13]:
"""
   h_rbf(z, w)

Given a flattened parameter vector `w` (of suitable size), return the value
of the complete RBF model.
"""
function h_rbf(z, w)
    dim_z = length(z)
    @assert dim_z > 0

    num_rbf_params = dim_z + 2 # (1 center + shape param + coefficient) per RBF “kernel”
    
    ## check length of `w` vector
    len_w = length(w)
    @assert len_w >= num_rbf_params
    @assert len_w % num_rbf_params == 0
    num_rbfs = div(len_w, num_rbf_params)

    ## unflatten and sum RBF kernels
    h_val = 0
    j = 1
    for i = 1:num_rbfs
        a = w[j]
        b = w[j+1]
        c = @view(w[j+2:j+1+dim_z])
        h_val += rbf(z, a, b, c)
        j += num_rbf_params
    end
    return h_val
end

h_rbf


<div class="alert alert-block alert-info">
Typically, models provided by some machine learning (like `Flux` or `Lux`), would not store 
their parameters in a flattened vector.
Not only is the model implementation more intuitive that way, it also allows for optimized
evaluation on large data sets.
For example, Flux models return `Params` objects, that usually store arrays of varying dimensions, 
dependent on the model structure.
The cool thing about `Zygote` is, that it keeps that structure when taking gradients.
So the gradient of some loss function with respect to parameters that are stored in a matrix 
is a matrix, which enables convenient updating.
<br/>
Both libraries offer tools for flattening to use external optimizers.
</div>

Now assume that $\mathbf Z \in ℝ^{n \times N}$ is a matrix, 
the columns of which hold feature vectors for labeled data.
The labels are in $\mathbf y\in ℝ^{N}$.
We want to use $h(•; \mathbf w)$ to approximate the data and the arrays induce a loss function 
$$
\mathcal L(\mathbf w) = \mathcal L(\mathbf w; \mathbf Z, \mathbf y) = \frac{1}{N}
\sum_{j=1}^{N}
(\mathbf y_j - h(\mathbf z_j, \mathbf w)) ^2
$$


Here is how to compute that value:

In [14]:
function rbf_mse(w, Z, y)
    mse_val = 0
    mse_divisor = 0
    for (zj, yj) = zip(eachcol(Z), y)
        mse_val += (h_rbf(zj, w) - yj)^2
        mse_divisor += 1
    end
    mse_val /= mse_divisor
    return mse_val
end

rbf_mse (generic function with 1 method)

#### Exercise

Now it's your turn.

Complete the cell below and use `Zygote.pullback` to compute the mean squared error **and the loss gradient**
with respect to $\mathbf w$ at the same time.
The syntax is 
```julia
result, back = Zygote.pullback( some_func, func_args )
```
and `back` is the **pullback function** that gives a tuple of gradient objects when called 
with seed `one(result)` (see lecture video on reverse-mode AD).

Oftentimes, `some_func` is actually anonymous, e.g., to compute partial gradients only.
In that case, it *can* be more convenient to use a `do`-block:
```julia
result, back = Zygote.pullback( func_arg_val1, func_arg_val2, … ) do func_arg_name1, func_arg_name2, …
    # function body acting on `func_arg_name1` etc.
    local_result
end
```
Here is the exercise:

In [ ]:
function rbf_mse_and_grad(w, Z, y)
    # compute and return loss and loss gradient of `rbf_mse` with respect to w
    ### BEGIN SOLUTION

    ### END SOLUTION
end

rbf_mse_and_grad (generic function with 1 method)

Let's test the implementation:

In [16]:
import Random   ## for reproducible pseudo random numbers

In [17]:
let rng = Random.seed!(31415)
    n = 3
    N = 4

    ## random training data
    Z = rand(rng, n, N)
    y = rand(rng, N)

    ## build flattened parameter vector
    num_rbfs = 10
    w = rand(rng, (n + 2) * num_rbfs)

    ## compute loss on random data
    _L = rbf_mse(w, Z, y)

    ## compute loss and loss gradient
    L, dL = rbf_mse_and_grad(w, Z, y)

    @assert L isa Number
    @assert L == _L
    @assert dL isa Vector

    ## check consistency with ForwardDiff
    _dL = FD.gradient(_w -> rbf_mse(_w, Z, y), w)
    @assert _dL ≈ dL

    ### BEGIN HIDDEN TESTS
    dim_data = 6
    num_data = 6

    Z = zeros(dim_data, num_data)
    y = zeros(num_data)

    n_params = (dim_data + 2) * num_rbfs
    w = zeros(n_params)
    for i = eachindex(w)
        if i % (dim_data + 2) == 1
            w[i] = 1
        end
    end

    _L = rbf_mse(w, Z, y)
    L, dL = rbf_mse_and_grad(w, Z, y)
    @assert _L == L
    @assert all(abs.(dL) .< 1e-5)
    ### END HIDDEN TESTS
end

## Exercise 2 - Momentum Descent

By extending the classical (stochastic) gradient descent update rule by a momentum term,
we can sometimes observe an improved convergence rate.
Polyak's Heavy Ball momentum is one of the best-known examples of this class of algorithms.
The parameter update for a loss 
$\mathcal L\colon ℝ^q \to ℝ, \mathbf w \mapsto \mathcal L(\mathbf w)$ 
in iteration $k\in ℕ_0$ is 
$$
\mathbf w^{(k+1)}
\leftarrow
\mathbf w^{(k)}
-
α \nabla \mathcal L(\mathbf w^{(k)})
+ 
κ
(\mathbf w^{(k)} - \mathbf w^{(k-1)}).
$$
We assume to start with $\mathbf w^{(0)}$.
The algorithm is instantiated with $\mathbf w^{(-1)} = \mathbf w^{(0)}$, i.e., without momentum in the first iteration.

To test our algorithm, we are again considering the _Rosenbrock function_:
$$
\mathcal L(\mathbf w) = (a - w_1)^2 + b(w_2 - w_1^2)^2.
$$

In [18]:
function rosenbrock_2D(w; a=1, b=100)
    return (a - w[1])^2 + b * (w[2] - w[1]^2)^2
end

rosenbrock_2D (generic function with 1 method)

### Exercise 2a)
Implement the **exact** gradient of the Rosenbrock function. You can compute it by hand or use `ForwardDiff` or `Zygote`.

In [ ]:
function dw_rosenbrock_2D(w; a=1, b=100)
    ### BEGIN SOLUTION
   
    ### END SOLUTION
end

dw_rosenbrock_2D (generic function with 1 method)

The global optimum is known to be $\mathbf x = [a, a^2]^\top$ and we can check for consistency,
if the value is zero and the gradient vanishes:

In [20]:
let # introduce a local scope to not pollute the global scope by accident 
    L = rosenbrock_2D([1, 1])
    @assert iszero(L)
    dL = dw_rosenbrock_2D([1, 1])
    @assert all(iszero.(dL))

    ### BEGIN HIDDEN TESTS
    a = rand()
    L = rosenbrock_2D([a, a^2]; a)
    @assert iszero(L)
    dL = dw_rosenbrock_2D([a, a^2]; a)
    @assert all(iszero.(dL))

    w = rand(2)
    a = rand()
    b = rand()
    dL = dw_rosenbrock_2D(w; a, b)
    @assert dL ≈ FD.gradient(_w -> rosenbrock_2D(_w; a, b), w)
    ### END HIDDEN TESTS
end

### Exercise 2b)

We now want to implement a training algorithm `momentum_descent` with fixed
meta-parameters.
Besides the meta parameters, the function is provided with functions
`loss`, `dw_loss` and initial parameters `w`.

Assume `loss` to return a scalar loss value when called as `loss(w)`, and `dw_loss`
to return a loss gradient vector.

Fill in the cell below according to the comments to finalize the algorithm implementation:

In [ ]:
function momentum_descent(
    loss,           # loss function
    dw_loss,        # gradient function
    w;              # initial model parameters
    num_iter=10,    # number of iterations              
    alpha=0.1f0,    # gradient stepsize
    kappa=0.1f0,    # momentum factor
)

    wk = copy(w)  # do not modify the initial vector!

    ## It is good practice to preallocate memory before any loop.
    ## Hence, initialize an object `w_prev` for the previous parameters and set 
    ## the values according to the formula for the Heavy Ball scheme.
    ## Additionally, allocate a vector `momentum` for the momentum term,
    ## i.e., the difference between two consecutive weight vectors.
    ### BEGIN SOLUTION
   
    ### END SOLUTION

    ## Finally, do the iterations by completing the code in the for loop below.
    ## Right now, `k` can be thought to equal 0.
    ## At the end of each loop iteration, `wk` should be updated to be valid for `k`.
    ## E.g., if `num_iter==1`, we have 1 iteration, update `wk`, and return ``\mathbf w^{(1)}``.
    for k = 1:num_iter
        ## 1) Obtain a loss gradient for the current `wk`.
        ## 2) Modify `momentum`, `w_prev` and `wk` according to the Heavy Ball formulas.
        ### BEGIN SOLUTION
 
        ### END SOLUTION
    end

    ## return last parameters
    return wk
end

momentum_descent (generic function with 1 method)

### Exercise 2c)

Minimize the Rosenbrock function with $a=1$ and $b=100$ using your momentum algorithm. Use the same method to compare the results to those of the standard gradient descent (think on how to choose the hyper-parameters to achieve this!)

Perform 100 iterations, starting at `w0_rb`:

In [ ]:
num_iter_rb = 100

## some initial guess:
w0_rb = [-π / 2, ℯ / 4] # don't change!

## obtain loss function from `rosenbrock_2D` for `a=1` and `b=100` and assign it to `loss_rb`.
## `loss_rb` will be the first argument for `momentum_descent`
loss_rb = nothing    # redefine below
### BEGIN SOLUTION

### END SOLUTION

## likewise, obtain loss gradient function `dw_loss_rb` from `rosenbrock_2D` for `a=1` and `b=100`
## `dw_loss_rb` will be the second argument for `momentum_descent`
dw_loss_rb = nothing   # redefine below
### BEGIN SOLUTION

### END SOLUTION

## here you can see how we do 100 iterations of momentum descent:
alpha_momentum = 1e-3
kappa_momentum = 0.7
wopt_rb_momentum = momentum_descent(
    loss_rb, dw_loss_rb, w0_rb;
    num_iter=num_iter_rb, alpha=alpha_momentum, kappa=kappa_momentum
)

## exercise: compare against steepest descent!
alpha_sd = alpha_momentum
kappa_sd = nothing  # redefine below
### BEGIN SOLUTION

### END SOLUTION
wopt_rb_sd = momentum_descent(
    loss_rb, dw_loss_rb, w0_rb;
    num_iter=num_iter_rb, alpha=alpha_sd, kappa=kappa_sd
);

Let's check your results:

In [23]:
@assert kappa_sd isa Real
### BEGIN HIDDEN TESTS
@assert iszero(kappa_sd)
### END HIDDEN TESTS

In [24]:
@assert wopt_rb_momentum isa Vector
@assert wopt_rb_sd isa Vector

loss_rb(w0_rb) ≈ 326.24283461518615
dw_loss_rb(w0_rb) ≈ [-1128.4687155349027; -357.56612863151565]

# After iteration, the loss has hopefully been reduced:
@assert loss_rb(wopt_rb_sd) < loss_rb(w0_rb)
@assert loss_rb(wopt_rb_momentum) < loss_rb(w0_rb)
### BEGIN HIDDEN TESTS
@assert loss_rb(wopt_rb_momentum) < loss_rb(wopt_rb_sd)
### END HIDDEN TESTS

If your implementation works as it should, the solution trajectories look something like this:
![momentum descent with rosenbrock](momentum_rb.png)

### Exercise 2d)

Finally, in the cell below, test several configurations of the descent algorithm for 
optimization of the Rosenbrock function $\mathcal L$ with $a=1$ and $b=100$.

For `w0_rb` from above, perform 50 iterations of momentum descent for the 
meta-parameters $(α, κ)$ in `alpha_kappa_configs`.
Among those tuples, determine the tuple `(alpha_best, kappa_best)` that achieves the smallest
function value after 50 iterations.


In [ ]:
## these are the meta-parameters to investigate
alpha_kappa_configs = Iterators.product((1e-4, 1e-5, 1e-6, 1e-7), (0.6, 0.2, 0.1, 1e-3, 1e-4, 1e-6))

## at the end of this cell block, you should have assigned fitting values to this tuple:
(alpha_best, kappa_best) = (-1.0, -1.0)
### BEGIN SOLUTION

### END SOLUTION

(0.0001, 0.6)

In [26]:
@assert alpha_best isa Real
@assert kappa_best isa Real
### BEGIN HIDDEN TESTS
@assert (alpha_best, kappa_best) == (1e-4, 0.6)
### END HIDDEN TESTS

## Exercise 3 - Newton's Method

Newton's method is a root finding algorithm.
Consider the function $\mathbf m\colon ℝ^q \to ℝ^q$.
Then Newton's method tries to find $\mathbf w^* \in ℝ^q$ with $\mathbf m(\mathbf w^*) = \mathbf 0$.
The update rule is 
$$
\mathbf w_{k+1} = \mathbf w_k - \left(\nabla \mathbf m(\mathbf w_k)\right)^{-1} \mathbf m(\mathbf w_k),
\quad 
k\in \mathbb N_0,
$$
where $\nabla \mathbf m(\mathbf w_k)$ is the Jacobian of $\mathbf m$ at $\mathbf w_k$.

Instead of computing the inverse in the right-most term, rather substitute it by $\mathbf v$ and solve a linear equation system
$$
\nabla \mathbf m(\mathbf w_k) \mathbf v = - \mathbf m(\mathbf w_k)
$$
to obtain $\mathbf w_{k+1} = \mathbf v + \mathbf w_k$.

### Exercise 3a)

In the cell below, use the formulas from above to implement Newton's root finding algorithm:

In [ ]:
function newton_opt(
    func,       # the function ``m`` from above
    jac_func,   # a function to obtain the jacobian of ``m``
    w0          # initial guess for the root
    ;
    num_iter=10,    # number of iterations
    abs_tol=0,      # absolute stopping tolerance
)
    ## initialize `wk`
    wk = copy(w0)
    
    for k=1:num_iter
        ## evaluate `func` at `wk` and assign results to `mk`
        ### BEGIN SOLUTION
        
        ### END SOLUTION
        
        ## if norm squared of `mk` is <= `abs_tol`, then break:
        ### BEGIN SOLUTION
      
        ### END SOLUTION

        ## compute jacobian at `wk` and store result in `d_mk`
        ## then set `v` to solve `d_mk * v = -mk`
        ### BEGIN SOLUTION
       
        ### END SOLUTION

        ## finally, update `wk` with `v` to store values for iteration `k+1`
        ### BEGIN SOLUTION
       
        ### END SOLUTION
    end

    return wk
end

newton_opt (generic function with 1 method)

Let us test your algorithm with a simple example. 
The Jacobian of $\mathbf m( w_1, w_2 ) = [w_1^2, w_2^2]$ is 
$$
\begin{bmatrix}
    2w_1 & 0 \\
    0 & 2w_2
\end{bmatrix}.
$$
Newton's algorithm should approximate $\mathbf w^* = [0, 0]^T$:

In [28]:
let
    ## test function:
    m = w -> w .^ 2
    ## jacobian:
    dw_m = w -> [
        2*w[1] 0;
        0 2*w[2]
    ]
    
    ## inital guess
    w0 = [1.0, -3.0]
    ## call newton root finder:
    wopt = newton_opt(m, dw_m, w0; num_iter=20, abs_tol=0)
    @assert sum(m(wopt) .^ 2) <= 1e-10

    ## test stopping criterion:
    _wopt = newton_opt(m, dw_m, w0; num_iter=20, abs_tol=1e-3)
    @assert sum(m(_wopt) .^ 2) <= 1e-3
    @assert m(wopt) <= m(wopt)
end

### Exercise 3b)

We want to use Newton's method to **minimize** the Rosenbrock function.
As an optimization algorithm for $\mathcal L\colon ℝ^q \to ℝ$, Newton's root finding scheme is applied to the function 
$$
\mathbf w \mapsto \nabla \mathcal L(\mathbf w)
$$
Hence, we need the Jacobian of the gradient of the Rosenbrock function for `newton_opt`, which is the _Hessian matrix_.
Complete the cell below to return this matrix:

In [ ]:
function hess_rosenbrock_2D(w; a=1, b=100)
    ### BEGIN SOLUTION
    
    ### END SOLUTION
end

hess_rosenbrock_2D (generic function with 1 method)

In [30]:
@assert hess_rosenbrock_2D(ones(2)) ≈ [
    802 -400;
    -400 200
]
### BEGIN HIDDEN TESTS
let rng = Random.seed!(1618), w = rand(rng, 2)
    @assert hess_rosenbrock_2D(w; a=2, b=50) ≈ FD.jacobian(_w -> dw_rosenbrock_2D(_w; a=2, b=50), w)
end
### END HIDDEN TESTS

Now apply `newton_opt` to the problem of minimizing the Rosenbrock function:

In [ ]:
m_rb = nothing
## below, override `m_rb` to a suitable function, such that Newton's method optimizes the
## Rosenbrock function with parameters `a = 2, b = 100`. 
## `m_rb` is given to `newton_opt` as the first argument.
### BEGIN SOLUTION

### END SOLUTION 

dw_m_rb = nothing
## below, set `dw_m_rb` to a suitable function that is used as the second argument for `newton_opt`:
### BEGIN SOLUTION

### END SOLUTION

## let's actually call the algorithm:
w_opt_newton = newton_opt(m_rb, dw_m_rb, w0_rb);

In [32]:
@assert w_opt_newton isa Vector
@assert length(w_opt_newton) == 2
### BEGIN HIDDEN TESTS
@assert w_opt_newton ≈ [2, 4]
### END HIDDEN TESTS

Now, the solution trajectory should reach the optimum quickly:
![newton iterations](newton.png)